In [7]:
pip install imblearn optuna-integration

   ---------------------------------------- 0.0/93.4 kB ? eta -:--:--
   ---------------------------------------- 93.4/93.4 kB 1.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import optuna
from tqdm import tqdm

n_trials = 100


def do_the_thing(df, filename):

    X = df.drop('Label', axis=1)
    y = df['Label']

    # Preprocess with SMOTE
    smote = SMOTE()
    X_res, y_res = smote.fit_resample(X, y)

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

    def tune_random_forest(X_train, y_train):

        def objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 200),
                'max_depth': None,
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 5),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
                'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
                'criterion': trial.suggest_categorical('criterion', ['entropy', 'gini']),
                'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced'])
            }
            model = RandomForestClassifier(bootstrap=True, **params, random_state=42)
            skf = StratifiedKFold(n_splits=5)
            return cross_val_score(model, X_train, y_train, cv=skf, scoring='f1').mean()
        
        study = optuna.create_study(direction='maximize')
        with tqdm(total=100, desc=f"Optimizing Random Forest for {filename}") as pbar:
            def callback(study, trial):
                pbar.update(1)
            
            study.optimize(objective, n_trials=100, callbacks=[callback])
        return study.best_params


    def tune_logistic_regression(X_train, y_train):

        def objective(trial):
            params = {
                'C': trial.suggest_loguniform('C', 0.001, 100),
                'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced']),
                'penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
                'solver': trial.suggest_categorical('solver', ['liblinear', 'saga'])
            }
            model = LogisticRegression(max_iter=1000, **params, random_state=42)
            skf = StratifiedKFold(n_splits=5)
            return cross_val_score(model, X_train, y_train, cv=skf, scoring='f1').mean()
        
        study = optuna.create_study(direction='maximize')
        with tqdm(total=100, desc=f"Optimizing Logistic Regression for {filename}") as pbar:
            def callback(study, trial):
                pbar.update(1)
            
            study.optimize(objective, n_trials=100, callbacks=[callback])
        return study.best_params


    def tune_neural_network(X_train, y_train):

        def objective(trial):
            hidden_layer_sizes = trial.suggest_categorical(
                'hidden_layer_sizes', [
                    (256, 128, 128, 64, 64),
                    (256, 128, 64, 64, 32),
                    (256, 128, 64, 32, 32),
                    (256, 128, 64, 32, 16)
                ])
            params = {
                'hidden_layer_sizes': hidden_layer_sizes,
                'activation': trial.suggest_categorical('activation', ['relu', 'tanh', 'logistic']),
                'solver': 'adam',
                'batch_size': trial.suggest_categorical('batch_size', [64, 128, 256, 512, 1024]),
                'learning_rate': trial.suggest_categorical('learning_rate', ['constant', 'invscaling', 'adaptive']),
                'learning_rate_init': trial.suggest_loguniform('learning_rate_init', 0.001, 0.1)
            }
            model = MLPClassifier(max_iter=1000, **params, random_state=42)
            skf = StratifiedKFold(n_splits=5)
            return cross_val_score(model, X_train, y_train, cv=skf, scoring='f1').mean()
            
        study = optuna.create_study(direction='maximize')
        with tqdm(total=100, desc=f"Optimizing Neural Network for {filename}") as pbar:
            def callback(study, trial):
                pbar.update(1)
            
            study.optimize(objective, n_trials=100, callbacks=[callback])
        return study.best_params


    def train_and_evaluate(model, X_train, y_train, X_test, y_test):
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        return {
            'F1-Score': f1_score(y_test, predictions),
            'Accuracy': accuracy_score(y_test, predictions),
            'Confusion Matrix': confusion_matrix(y_test, predictions)
        }

    # Tuning and evaluating each model
    rf_params = tune_random_forest(X_train, y_train)
    lr_params = tune_logistic_regression(X_train, y_train)
    nn_params = tune_neural_network(X_train, y_train)

    # Model setup
    rf_model = RandomForestClassifier(bootstrap=True, **rf_params, random_state=42)
    lr_model = LogisticRegression(max_iter=1000, **lr_params, random_state=42)
    nn_model = MLPClassifier(solver='adam', max_iter=1000, **nn_params, random_state=42)

    # Performance evaluation
    rf_performance = train_and_evaluate(rf_model, X_train, y_train, X_test, y_test)
    lr_performance = train_and_evaluate(lr_model, X_train, y_train, X_test, y_test)
    nn_performance = train_and_evaluate(nn_model, X_train, y_train, X_test, y_test)

    return rf_performance, lr_performance, nn_performance



In [9]:
import pandas as pd
import os

# Function definitions (assuming do_the_thing is imported or defined in the script)

# Assume this list of filenames represents the CSV files in the folder
filenames = ["all-MiniLM-L6-v2.csv", "all-mpnet-base-v2.csv", "gtr-t5-base.csv", "sentence-t5-base.csv"]

# Prepare a dictionary to store results
all_results = {
    'Filename': [],
    'Model': [],
    'F1-Score': [],
    'Accuracy': [],
    'Confusion Matrix': []
}

# Loop through each file and apply the do_the_thing function
for filename in filenames:
    df = pd.read_csv(filename)
    rf_performance, lr_performance, nn_performance = do_the_thing(df, filename)
    print(rf_performance, lr_performance, nn_performance)
    
    # Append results to the dictionary
    for model_name, performance in zip(['Random Forest', 'Logistic Regression', 'Neural Network'],
                                       [rf_performance, lr_performance, nn_performance]):
        all_results['Filename'].append(filename)
        all_results['Model'].append(model_name)
        all_results['F1-Score'].append(performance['F1-Score'])
        all_results['Accuracy'].append(performance['Accuracy'])
        all_results['Confusion Matrix'].append(performance['Confusion Matrix'])

# Convert the results dictionary into a DataFrame
results_df = pd.DataFrame(all_results)

# Print the DataFrame to see the results
print(results_df)

# Save the DataFrame to a new CSV file
results_df.to_csv("combined_results.csv", index=False)


[I 2024-05-09 00:38:58,286] A new study created in memory with name: no-name-6ab8ea63-d01f-41da-842d-acb121164abf
Optimizing Random Forest for all-MiniLM-L6-v2.csv:   0%|          | 0/100 [00:00<?, ?it/s][I 2024-05-09 00:39:03,847] Trial 0 finished with value: 0.948872665516473 and parameters: {'n_estimators': 165, 'min_samples_split': 3, 'min_samples_leaf': 2, 'max_features': 'log2', 'criterion': 'gini', 'class_weight': None}. Best is trial 0 with value: 0.948872665516473.
Optimizing Random Forest for all-MiniLM-L6-v2.csv:   1%|          | 1/100 [00:05<09:08,  5.54s/it][I 2024-05-09 00:39:07,793] Trial 1 finished with value: 0.9450512467665385 and parameters: {'n_estimators': 119, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'criterion': 'gini', 'class_weight': None}. Best is trial 0 with value: 0.948872665516473.
Optimizing Random Forest for all-MiniLM-L6-v2.csv:   2%|▏         | 2/100 [00:09<07:31,  4.60s/it][I 2024-05-09 00:39:11,506] Trial 2 finished with

{'F1-Score': 0.9710144927536232, 'Accuracy': 0.9714285714285714, 'Confusion Matrix': array([[138,   2],
       [  6, 134]], dtype=int64)} {'F1-Score': 0.9550173010380623, 'Accuracy': 0.9535714285714286, 'Confusion Matrix': array([[129,  11],
       [  2, 138]], dtype=int64)} {'F1-Score': 0.9614035087719298, 'Accuracy': 0.9607142857142857, 'Confusion Matrix': array([[132,   8],
       [  3, 137]], dtype=int64)}


[I 2024-05-09 01:11:41,617] A new study created in memory with name: no-name-5427d224-b963-4714-8822-2a8b7953d09b
Optimizing Random Forest for all-mpnet-base-v2.csv:   0%|          | 0/100 [00:00<?, ?it/s][I 2024-05-09 01:11:56,493] Trial 0 finished with value: 0.9668780326889032 and parameters: {'n_estimators': 181, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'criterion': 'entropy', 'class_weight': None}. Best is trial 0 with value: 0.9668780326889032.
Optimizing Random Forest for all-mpnet-base-v2.csv:   1%|          | 1/100 [00:14<24:32, 14.88s/it][I 2024-05-09 01:12:01,591] Trial 1 finished with value: 0.9647934606203761 and parameters: {'n_estimators': 151, 'min_samples_split': 3, 'min_samples_leaf': 4, 'max_features': 'log2', 'criterion': 'entropy', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.9668780326889032.
Optimizing Random Forest for all-mpnet-base-v2.csv:   2%|▏         | 2/100 [00:19<14:54,  9.12s/it][I 2024-05-09 01:12:07,941] Tri

{'F1-Score': 0.9819494584837546, 'Accuracy': 0.9821428571428571, 'Confusion Matrix': array([[139,   1],
       [  4, 136]], dtype=int64)} {'F1-Score': 0.9823321554770317, 'Accuracy': 0.9821428571428571, 'Confusion Matrix': array([[136,   4],
       [  1, 139]], dtype=int64)} {'F1-Score': 0.9824561403508771, 'Accuracy': 0.9821428571428571, 'Confusion Matrix': array([[135,   5],
       [  0, 140]], dtype=int64)}


[I 2024-05-09 01:57:51,224] A new study created in memory with name: no-name-25fd11ff-7afe-481b-862f-9bb044de6b38
Optimizing Random Forest for gtr-t5-base.csv: 100%|██████████| 100/100 [18:45<00:00, 11.26s/it]
[I 2024-05-09 02:16:36,831] A new study created in memory with name: no-name-957ffa52-e304-4a79-9b75-6024f368fe05
Optimizing Logistic Regression for gtr-t5-base.csv:   0%|          | 0/100 [00:00<?, ?it/s]C:\Users\madha\AppData\Local\Temp\ipykernel_3832\2556245725.py:55: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'C': trial.suggest_loguniform('C', 0.001, 100),
C:\Users\madha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warn

{'F1-Score': 0.9747292418772563, 'Accuracy': 0.975, 'Confusion Matrix': array([[138,   2],
       [  5, 135]], dtype=int64)} {'F1-Score': 0.9679715302491102, 'Accuracy': 0.9678571428571429, 'Confusion Matrix': array([[135,   5],
       [  4, 136]], dtype=int64)} {'F1-Score': 0.9752650176678446, 'Accuracy': 0.975, 'Confusion Matrix': array([[135,   5],
       [  2, 138]], dtype=int64)}


[I 2024-05-09 03:29:07,898] A new study created in memory with name: no-name-13b38eff-94b3-4298-b9f1-b606b76c4cb4
Optimizing Random Forest for sentence-t5-base.csv:   0%|          | 0/100 [00:00<?, ?it/s][I 2024-05-09 03:29:19,154] Trial 0 finished with value: 0.9689599966585684 and parameters: {'n_estimators': 195, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'criterion': 'gini', 'class_weight': None}. Best is trial 0 with value: 0.9689599966585684.
Optimizing Random Forest for sentence-t5-base.csv:   1%|          | 1/100 [00:11<18:34, 11.25s/it][I 2024-05-09 03:29:24,960] Trial 1 finished with value: 0.9724725418222764 and parameters: {'n_estimators': 176, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 'log2', 'criterion': 'entropy', 'class_weight': None}. Best is trial 1 with value: 0.9724725418222764.
Optimizing Random Forest for sentence-t5-base.csv:   2%|▏         | 2/100 [00:17<13:08,  8.05s/it][I 2024-05-09 03:29:26,626] Trial 2 finishe

{'F1-Score': 0.9785714285714285, 'Accuracy': 0.9785714285714285, 'Confusion Matrix': array([[137,   3],
       [  3, 137]], dtype=int64)} {'F1-Score': 0.9750889679715302, 'Accuracy': 0.975, 'Confusion Matrix': array([[136,   4],
       [  3, 137]], dtype=int64)} {'F1-Score': 0.9784172661870504, 'Accuracy': 0.9785714285714285, 'Confusion Matrix': array([[138,   2],
       [  4, 136]], dtype=int64)}
                 Filename                Model  F1-Score  Accuracy  \
0    all-MiniLM-L6-v2.csv        Random Forest  0.971014  0.971429   
1    all-MiniLM-L6-v2.csv  Logistic Regression  0.955017  0.953571   
2    all-MiniLM-L6-v2.csv       Neural Network  0.961404  0.960714   
3   all-mpnet-base-v2.csv        Random Forest  0.981949  0.982143   
4   all-mpnet-base-v2.csv  Logistic Regression  0.982332  0.982143   
5   all-mpnet-base-v2.csv       Neural Network  0.982456  0.982143   
6         gtr-t5-base.csv        Random Forest  0.974729  0.975000   
7         gtr-t5-base.csv  Logistic Reg